# Gold — Agregação para Análise

Este notebook constrói a camada **gold**: lê os dados limpos da silver (`growdev.silver`), junta as tabelas com `join` e cria uma tabela agregada **pronta para análise de negócio** em `growdev.gold.resumo_clientes`. Enquanto a bronze guarda o dado bruto e a silver o dado limpo, a gold guarda o dado **resumido** — é a camada consultada por dashboards e relatórios. No caminho, recapitulamos join, groupBy, agg, filter, orderBy e write.

In [0]:
# Recapitulando: variáveis e f-strings
from pyspark.sql import functions as F

CATALOG = "growdev"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"

# Lê as duas tabelas da silver que vamos usar
df_loans = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.loans")
df_customers = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.customers")

print(f"loans: {df_loans.count()} linhas")
print(f"customers: {df_customers.count()} linhas")

In [0]:
# Recapitulando: join + groupBy + agg + write

# 1. Junta empréstimos com clientes pelo customer_id
# 2. Agrupa por cliente e calcula as métricas
df_gold = (
    df_loans.join(df_customers, "customer_id")
    .groupBy("customer_id", "full_name", "state", "customer_segment")
    .agg(
        F.count("loan_id").alias("qtd_emprestimos"),
        F.sum("loan_amount").alias("valor_total_emprestado"),
        F.sum(F.when(F.col("is_default"), 1).otherwise(0)).alias("qtd_inadimplencias"),
    )
)

tabela_gold = f"{CATALOG}.{SCHEMA_GOLD}.resumo_clientes"  # ex.: growdev.gold.resumo_clientes

# A gold é sempre recalculada por completo: overwrite substitui o resultado anterior
df_gold.write.mode("overwrite").saveAsTable(tabela_gold)

print(f"OK: {tabela_gold} criada com {df_gold.count()} clientes")

In [0]:
# Recapitulando: select, filter, orderBy e groupBy

# 1) Lê a tabela gold recém-criada
df_resumo = spark.table(tabela_gold)

# 2) Clientes com maior valor total emprestado
display(df_resumo.orderBy(F.desc("valor_total_emprestado")).limit(10))

# 3) Inadimplência média por segmento de cliente
resumo_segmento = (
    df_resumo.groupBy("customer_segment")
    .agg(
        F.count("*").alias("clientes"),
        F.avg("qtd_inadimplencias").alias("inadimplencias_media"),
    )
    .orderBy(F.desc("clientes"))
)
display(resumo_segmento)